In [ ]:
!pip install openpyxl
import pandas as pd
import numpy as np
import io
import os
import zipfile
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

# Configuration esthétique des graphiques
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("--- CHARGEMENT DU DOSSIER ---")
# Détection automatique de votre fichier s'il est déjà dans l'environnement
fichiers_presents = os.listdir('.')
archive_zip = [f for f in fichiers_presents if f.endswith('.zip')]

if archive_zip:
    file_name = archive_zip[0]
    print(f"✓ Archive détectée : {file_name}")
    with zipfile.ZipFile(file_name, 'r') as z:
        csv_interne = [f for f in z.namelist() if f.endswith('.csv')][0]
        with z.open(csv_interne) as f:
            df = pd.read_csv(io.BytesIO(f.read()), encoding='latin-1')
else:
    uploaded = files.upload()
    file_name = list(uploaded.keys())[0]
    if file_name.endswith('.zip'):
        with zipfile.ZipFile(io.BytesIO(uploaded[file_name]), 'r') as z:
            csv_interne = [f for f in z.namelist() if f.endswith('.csv')][0]
            with z.open(csv_interne) as f:
                df = pd.read_csv(io.BytesIO(f.read()), encoding='latin-1')
    elif file_name.endswith('.csv'):
        df = pd.read_csv(io.BytesIO(uploaded[file_name]), encoding='latin-1')
    elif file_name.endswith(('.xls', '.xlsx')):
        df = pd.read_excel(io.BytesIO(uploaded[file_name]))

print(f"Chargement réussi ! Dimensions : {df.shape[0]} lignes, {df.shape[1]} colonnes\n")

# =====================================================================
# Q1 : États avec le plus de ventes & Q4 : Différences de rentabilité
# =====================================================================
print("--- Q1 & Q4 : ANALYSE DES ÉTATS (VENTES ET RENTABILITÉ) ---")
etats_stats = df.groupby('State').agg({'Sales': 'sum', 'Profit': 'sum'}).sort_values(ascending=False, by='Sales')
etats_stats['Marge_%'] = (etats_stats['Profit'] / etats_stats['Sales']) * 100

# Graphique Ventes par État (Top 15)
plt.figure(figsize=(12, 5))
sns.barplot(x=etats_stats['Sales'].head(15).values, y=etats_stats['Sales'].head(15).index, palette='Blues_r')
plt.title("Top 15 des États par Chiffre d'Affaires", fontsize=13, fontweight='bold')
plt.xlabel("Ventes Totales ($)")
plt.show()

# Affichage des écarts de rentabilité (Marge négative = déficit)
print("\nFocus sur la rentabilité de quelques États clés (Différences flagrantes) :")
print(etats_stats[['Sales', 'Profit', 'Marge_%']].head(10).to_string())
print("\n" + "="*80 + "\n")

# =====================================================================
# Q2 : Comparaison stricte NY vs Californie
# =====================================================================
print("--- Q2 : COMPARAISON NEW YORK VS CALIFORNIA ---")
comp_ny_ca = df[df['State'].isin(['New York', 'California'])].groupby('State')[['Sales', 'Profit']].sum()
comp_ny_ca['Marge_%'] = (comp_ny_ca['Profit'] / comp_ny_ca['Sales']) * 100
print(comp_ny_ca.to_string())

comp_ny_ca[['Sales', 'Profit']].plot(kind='bar', color=['skyblue', 'salmon'], edgecolor='black')
plt.title("Comparaison Financière : New York vs Californie", fontsize=13, fontweight='bold')
plt.ylabel("Montant ($)")
plt.xticks(rotation=0)
plt.show()
print("\n" + "="*80 + "\n")

# =====================================================================
# Q3 : Client exceptionnel à New York
# =====================================================================
print("--- Q3 : CLIENT EXCEPTIONNEL À NEW YORK ---")
ny_data = df[df['State'] == 'New York']
client_ny = ny_data.groupby('Customer Name').agg({'Sales': 'sum', 'Profit': 'sum'}).sort_values(by='Profit', ascending=False)
print("Le client le plus performant et exceptionnel (générant le plus de profit) à NY est :")
print(client_ny.head(1))
print("\n" + "="*80 + "\n")

# =====================================================================
# Q5 : Principe de Pareto appliqué aux Clients et aux PROFITS
# =====================================================================
print("--- Q5 : PARETO - CLIENTS VS PROFITS ---")
# On filtre les profits positifs pour l'analyse Pareto classique
clients_profit = df.groupby('Customer Name')['Profit'].sum().sort_values(ascending=False).reset_index()
clients_profit = clients_profit[clients_profit['Profit'] > 0]
clients_profit['Cumul_Profit'] = clients_profit['Profit'].cumsum()
total_profit = clients_profit['Profit'].sum()
clients_profit['Pct_Cumul_Profit'] = clients_profit['Cumul_Profit'] / total_profit
clients_profit['Pct_Clients'] = (clients_profit.index + 1) / len(clients_profit)

plt.figure(figsize=(10, 5))
plt.plot(clients_profit['Pct_Clients'].values, clients_profit['Pct_Cumul_Profit'].values, color='green', linewidth=2)
plt.axhline(y=0.8, color='r', linestyle='--', label='80% des Profits')
plt.axvline(x=0.2, color='orange', linestyle='--', label='20% des Clients')
plt.title("Courbe de Pareto : Contribution des clients aux PROFITS", fontsize=13, fontweight='bold')
plt.xlabel("Proportion des clients (0 à 1)")
plt.ylabel("Proportion cumulée des profits")
plt.legend()
plt.show()

# Vérification du ratio exact
top_20_pct_clients = clients_profit[clients_profit['Pct_Clients'] <= 0.2]
print(f"Résultat : Les premiers 20% des clients représentent {top_20_pct_clients['Pct_Cumul_Profit'].iloc[-1]*100:.1f}% du profit total.")
print("\n" + "="*80 + "\n")

# =====================================================================
# Q6 : Top 20 Villes (Ventes vs Bénéfices) & Analyse Rentabilité
# =====================================================================
print("--- Q6 : TOP 20 VILLES (VENTES VS PROFITS) ---")
villes_stats = df.groupby('City').agg({'Sales': 'sum', 'Profit': 'sum'})
top_villes_sales = villes_stats.sort_values(by='Sales', ascending=False).head(20)
top_villes_profit = villes_stats.sort_values(by='Profit', ascending=False).head(20)

print("Top 5 Villes par Ventes :\n", top_villes_sales['Sales'].head(5))
print("\nTop 5 Villes par Profit :\n", top_villes_profit['Profit'].head(5))

# Analyse des anomalies de rentabilité urbaine
print("\nAnomalie détectée (Exemple de Philadelphie ou Houston si présentes) :")
villes_stats['Marge_%'] = (villes_stats['Profit'] / villes_stats['Sales']) * 100
print(villes_stats.sort_values(by='Sales', ascending=False)[['Sales', 'Profit', 'Marge_%']].head(10).to_string())
print("\n" + "="*80 + "\n")

# =====================================================================
# Q7 & Q8 : Top 20 Clients (Ventes) & Courbe cumulative Pareto Ventes
# =====================================================================
print("--- Q7 & Q8 : TOP 20 CLIENTS & PARETO VENTES ---")
clients_sales = df.groupby('Customer Name')['Sales'].sum().sort_values(ascending=False).reset_index()
print("Top 5 des meilleurs clients en termes de ventes globales :")
print(clients_sales.head(5).to_string())

clients_sales['Cumul_Sales'] = clients_sales['Sales'].cumsum()
total_sales = clients_sales['Sales'].sum()
clients_sales['Pct_Cumul_Sales'] = clients_sales['Cumul_Sales'] / total_sales
clients_sales['Pct_Clients'] = (clients_sales.index + 1) / len(clients_sales)

plt.figure(figsize=(10, 5))
plt.plot(clients_sales['Pct_Clients'].values, clients_sales['Pct_Cumul_Sales'].values, color='purple', linewidth=2)
plt.axhline(y=0.8, color='r', linestyle='--', label='80% des Ventes')
plt.axvline(x=0.2, color='orange', linestyle='--', label='20% des Clients')
plt.title("Courbe de Pareto : Contribution des clients au Chiffre d'Affaires (Ventes)", fontsize=13, fontweight='bold')
txt = "Proportion des clients (0 à 1)"
plt.xlabel(txt)
plt.ylabel("Proportion cumulée des ventes")
plt.legend()
plt.show()

print(f"Les premiers 20% des clients génèrent {clients_sales[clients_sales['Pct_Clients'] <= 0.2]['Pct_Cumul_Sales'].iloc[-1]*100:.1f}% des ventes totales.")